# 10A — Development error analysis

Run after the development evaluation. This notebook diagnoses a frozen run; it does not train, select, or open holdout. It reads development labels intentionally for error analysis.

Full mode replays the exact alert and evaluation functions, validates recorded lineage and outcomes, and stops on mismatch. Choose a **new output run ID** for each run. Large score/feature scans stream or use bounded DuckDB memory. Deep data scans may take several minutes on Drive.


In [ ]:
from pathlib import Path
import os
import sys

if "google.colab" in sys.modules:
    from google.colab import drive

    drive.mount("/content/drive")


def find_repository(start=Path.cwd()):
    """Find the checked-out repository when Jupyter starts in any subfolder."""
    override = os.getenv("TELCO_PROJECT_ROOT")
    if override:
        candidates = [Path(override).expanduser().resolve()]
    else:
        start = start.resolve()
        candidates = [start, *start.parents]
        if "google.colab" in sys.modules:
            candidates += [
                Path("/content/drive/MyDrive/anomaly_detection"),
                Path("/content/drive/MyDrive/telco-anomaly-detection"),
            ]
    for candidate in candidates:
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "configs"
        ).is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        "Open this notebook from the cloned repository, or set TELCO_PROJECT_ROOT."
    )


PROJECT_ROOT = find_repository()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from datetime import datetime, timezone
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from telco_anomaly.io import resolve_data_root
from telco_anomaly.diagnostics import run_diagnostics

DATA_ROOT = resolve_data_root()
RESULTS = DATA_ROOT / "results/synthetic_pon/synthetic_pon_development_v14"
CORE_RUN = "synthetic_pon_core_v2"
FEATURE_RUN = "synthetic_pon_features_v6"
TRUTH_RUN = "synthetic_pon_truth_v3"
FULL_MODE = True
DEEP_DATA = True  # Raw quality, collection gaps, feature missingness and shift.
TRACE_FAULT = "F-00023"  # Missed ONT hardware failure; set None to skip.
TRACE_CASE = None  # To review a nuisance incident, set its ID and TRACE_FAULT=None.
TRACE_LIMIT = 20000  # Per raw/feature/score table; truncation is reported.
RUN_ID = "development_diagnostics_" + datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S%fZ"
)
OUTPUT = DATA_ROOT / "diagnostics/synthetic_pon" / RUN_ID
print("Inputs:", RESULTS, "\nOutput:", OUTPUT)


## Generate the dossier

For an extracted results ZIP alone, set RESULTS to its development result folder and FULL_MODE=False. Full mode gets model and selection run IDs from the evaluation manifest; the explicit core/feature/truth IDs must match its hashes.


In [ ]:
run_diagnostics(
    RESULTS,
    OUTPUT,
    data_root=DATA_ROOT if FULL_MODE else None,
    core_run=CORE_RUN,
    feature_run=FEATURE_RUN,
    truth_run=TRUTH_RUN,
    trace_fault=TRACE_FAULT if FULL_MODE else None,
    trace_case=TRACE_CASE if FULL_MODE else None,
    trace_limit=TRACE_LIMIT,
    deep_data=DEEP_DATA and FULL_MODE,
)
display(Markdown((OUTPUT / "report.md").read_text()))


In [ ]:
faults = pd.read_parquet(OUTPUT / "fault_diagnostics.parquet")
display(faults)
display(pd.read_parquet(OUTPUT / "review_queue.parquet"))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
faults.groupby("fault_type").detected.mean().sort_values().plot.barh(ax=axes[0])
axes[0].set(xlabel="Development fault recall", xlim=(0, 1))
faults.loc[faults.detected, "delay_hours"].plot.hist(ax=axes[1], bins=15)
axes[1].set(
    xlabel="Detection delay (hours), detected faults only", ylabel="Fault count"
)
fig.tight_layout()
fig.savefig(OUTPUT / "fault_overview.png", dpi=160)
plt.show()


In [ ]:
if FULL_MODE:
    for name in [
        "incident_status_counts",
        "latency_recall_curve",
        "score_evidence",
        "candidate_frontier",
    ]:
        path = OUTPUT / f"{name}.parquet"
        if path.exists():
            display(Markdown(f"**{name}**"))
            display(pd.read_parquet(path))
    availability = pd.read_parquet(OUTPUT / "score_availability.parquet")
    display(availability.sort_values("valid_fraction_of_present_rows").head(30))
    if DEEP_DATA:
        quality = pd.read_parquet(OUTPUT / "feature_quality_and_shift.parquet")
        display(
            quality.sort_values("missing_fraction_development", ascending=False).head(
                40
            )
        )
        display(pd.read_parquet(OUTPUT / "raw_quality_by_metric.parquet"))


## Inspect one fault end to end

The exported trace includes context before the observable fault and is clipped to development. Check `trace_manifest` for truncation before interpreting it. Inspect raw measurements and feature response alongside the score: a subthreshold score alone cannot tell us whether the raw signal was absent, the feature was weak, or the model was insensitive.

For another fault, use `export_trace` into a **new empty directory** with the validated inputs from a new diagnostic run, or change TRACE_FAULT and rerun with a new RUN_ID. Avoid selecting future models on repeatedly reused development faults without fresh validation.


In [ ]:
import json

if FULL_MODE and TRACE_FAULT:
    display(pd.read_parquet(OUTPUT / "trace_manifest.parquet"))
    raw = pd.read_parquet(OUTPUT / "trace_raw.parquet")
    features = pd.read_parquet(OUTPUT / "trace_features.parquet")
    scores = pd.read_parquet(OUTPUT / "trace_scores.parquet")
    display(raw.head(30))
    display(features.head(20))
    row = faults.set_index("fault_id").loc[TRACE_FAULT]
    selected = json.loads(
        (
            DATA_ROOT
            / "selection/synthetic_pon"
            / json.loads((RESULTS / "evaluation_manifest.json").read_text())[
                "selection_run_id"
            ]
            / "selected_configuration.json"
        ).read_text()
    )
    fig, ax = plt.subplots(figsize=(12, 4))
    for channel in selected["channels"]:
        for entity, frame in scores.groupby("entity_id"):
            ax.plot(
                frame.event_ts, frame[channel], label=f"{entity}: {channel}", alpha=0.7
            )
        ax.axhline(
            selected["thresholds"][channel],
            linestyle="--",
            label=f"{channel} threshold",
        )
    for field in ["observable_ts", "impact_ts"]:
        if pd.notna(row[field]):
            ax.axvline(row[field], linestyle=":", label=field)
    ax.set(title=TRACE_FAULT, ylabel="Frozen anomaly score")
    ax.legend(loc="best")
    fig.autofmt_xdate()
    fig.tight_layout()
    fig.savefig(OUTPUT / "selected_fault_score.png", dpi=160)
    plt.show()


## Read the findings correctly

- `finite_scores_never_reached_frozen_threshold`: inspect raw/feature response; do not automatically lower the threshold.
- `threshold_crossing_without_alert`: investigate persistence, episode boundaries, gaps, and already-active alert state.
- `alert_opened_but_no_distinct_fault_credit`: inspect case membership and competing/overlapping faults.
- `shared_scope_reported_as_entity`: inspect selected channels and common-mode evidence at the incident opening.
- Duplicate and unmatched incidents require separate investigation. An unmatched incident is not proof of normal underlying telemetry.
- The latency curve reruns matching at each horizon; it is not obtained by filtering active-window matches.
- The candidate frontier does not authorise selection or bypass workload gates.

Reports are CSV + Parquet, with `report.md` and input fingerprints. This is a diagnostic baseline for designing experiments, not an automatic root-cause engine or a production qualification.
